# HC18 Historical Approach Ladder & 5-Fold Benchmark Evaluation

## 1. Purpose & Scope
This benchmark documents and verifies the evaluation of historical model iterations on the HC18 fetal head ultrasound challenge dataset. It compares early direct regression and multi-task models against segmentation-based U-Net architectures under a strictly leak-free patient-grouped protocol.

## 2. Benchmark Protocol
- **Grouped Fold 1 Evaluation (Rows A-E)**: Evaluated on the exact 206 held-out images from fold 1 of `make_grouped_folds(seed=42)`. Early stopping and hyperparameter tuning use an inner 10% patient-isolation validation slice partitioned strictly from training patients.
- **5-Fold Cross-Validation (Row D vs Row E)**: Row D (ResNet-34 U-Net at 224x224) trained and evaluated across all 5 folds (both with and without horizontal-flip TTA), compared against Row E (Stream 1 final model: EfficientNet-B4 U-Net at 384x384 with fitEllipseAMS) over all 999 images.
- **Paired Statistical Comparison**: Patient-level cluster bootstrap (1,000 resamples) computing 95% confidence intervals for the paired error differences ($\Delta \text{MAE} = \text{MAE}_D - \text{MAE}_E$), evaluated across all 999 images and excluding the known extreme outlier image `065_HC.png`.

## 3. Historical Approaches Evaluated
- **Row A**: Direct continuous scalar regression of (HC, BPD) using ConvNeXt-Tiny (224x224).
- **Row B**: Ellipse parameter regression $(x_c, y_c, a, b, \theta)$ with Huber loss using ConvNeXt-Tiny (224x224 letterbox).
- **Row C**: Multi-Task U-Net (ResNet-34 encoder + dual heads for segmentation mask and 5-parameter regression, 224x224).
- **Row D**: ResNet-34 U-Net (224x224 letterbox) predicting segmentation mask, fitted with robust native-resolution ellipse fitting.
- **Row E**: Stream 1 Final Model (EfficientNet-B4 U-Net, 384x384 letterbox, TTA, fitEllipseAMS).

> **Important Warning**: The early 224 px results reported in legacy exploration notebooks were obtained using random train/val image splits that had significant patient leakage (the same patient's ultrasound video frames appeared in both train and validation). When tested on an honest patient-grouped split, direct regression and geometry approaches exhibit significantly higher real error (~10-11 mm HC MAE), proving that segmentation followed by ellipse fitting is strictly required for clinical accuracy.

In [1]:
import json
from pathlib import Path
import pandas as pd

results_dir = None
for candidate in [
    Path('../../docs/results/hc18_ladder'),
    Path('work/docs/results/hc18_ladder'),
    Path('docs/results/hc18_ladder'),
    Path('/kaggle/working/out'),
]:
    if (candidate / 'ladder_results.json').exists():
        results_dir = candidate
        break

if results_dir is None:
    raise FileNotFoundError('Could not locate ladder_results.json')

with open(results_dir / 'ladder_results.json', 'r', encoding='utf-8') as f:
    ladder_data = json.load(f)

print('=== HC18 Historical Approach Ladder (Grouped Fold 1) ===')
rows = []
for m in ladder_data['metrics']:
    rows.append({
        'Model': m['model_name'].split('(')[0].strip(),
        'Backbone': m['backbone'],
        'Resolution': m['input_res'],
        'HC MAE (mm)': m['hc_mae'],
        'HC RMSE (mm)': m['hc_rmse'],
        'BPD MAE (mm)': m['bpd_mae'],
        'BPD RMSE (mm)': m['bpd_rmse'],
        'Epochs': m['epochs_trained'] if m.get('epochs_trained') is not None else '-',
        'Train Time (s)': m['train_time_sec'] if m.get('train_time_sec') is not None else '-',
    })
df_ladder = pd.DataFrame(rows)
print(df_ladder.to_string(index=False))

=== HC18 Historical Approach Ladder (Grouped Fold 1) ===
                           Model        Backbone          Resolution  HC MAE (mm)  HC RMSE (mm)  BPD MAE (mm)  BPD RMSE (mm) Epochs Train Time (s)
        Row A: Direct Regression   ConvNeXt-Tiny    224x224 (direct)      10.6728       15.9672        3.2616         4.7343     25         695.39
     Row B: Geometry Approach v3   ConvNeXt-Tiny 224x224 (letterbox)      11.5964       16.1775        3.2198         4.5392     20          307.4
         Row C: Multi-Task U-Net       ResNet-34    224x224 (direct)      10.5104       15.5098        2.9474         4.3227     52          548.1
          Row D: ResNet-34 U-Net       ResNet-34 224x224 (letterbox)       1.9246        2.4351        0.6003         0.8316     48         495.63
Row E: EfficientNet-B4 U-Net 384 EfficientNet-B4 384x384 (letterbox)       2.4817        5.0996        0.6982         0.9200      -              -


In [2]:
import json
from pathlib import Path
import pandas as pd

results_dir = None
for candidate in [
    Path('../../docs/results/hc18_ladder'),
    Path('work/docs/results/hc18_ladder'),
    Path('docs/results/hc18_ladder'),
    Path('/kaggle/working/out'),
]:
    if (candidate / 'd_5fold_results.json').exists():
        results_dir = candidate
        break

if results_dir is None:
    raise FileNotFoundError('Could not locate d_5fold_results.json')

with open(results_dir / 'd_5fold_results.json', 'r', encoding='utf-8') as f:
    d5_data = json.load(f)

print('=== Row D vs Row E 5-Fold Cross-Validation Summary ===')
print(f"Total pooled images: {d5_data['pooled']['n_images']}")
print(f"Row D (no TTA) Pooled HC MAE:   {d5_data['pooled']['d_no_tta']['hc_mae']:.4f} mm, BPD MAE: {d5_data['pooled']['d_no_tta']['bpd_mae']:.4f} mm")
print(f"Row D (+TTA)    Pooled HC MAE:   {d5_data['pooled']['d_tta']['hc_mae']:.4f} mm, BPD MAE: {d5_data['pooled']['d_tta']['bpd_mae']:.4f} mm")
print(f"Row E           Pooled HC MAE:   {d5_data['pooled']['e']['hc_mae']:.4f} mm, BPD MAE: {d5_data['pooled']['e']['bpd_mae']:.4f} mm")

print('\n--- Per-Fold Comparison ---')
fold_rows = []
for f in d5_data['per_fold']:
    fold_rows.append({
        'Fold': f['fold'],
        'N': f['n_images'],
        'D HC MAE': f['d_no_tta']['hc_mae'],
        'D+TTA HC MAE': f['d_tta']['hc_mae'],
        'E HC MAE': f['e']['hc_mae'],
        'D BPD MAE': f['d_no_tta']['bpd_mae'],
        'D+TTA BPD MAE': f['d_tta']['bpd_mae'],
        'E BPD MAE': f['e']['bpd_mae'],
    })
df_folds = pd.DataFrame(fold_rows)
print(df_folds.to_string(index=False))

print('\n--- Paired Difference (D - E) in HC MAE with Patient-Level Bootstrap 95% CI ---')
for key, label in [('d_no_tta_vs_e', 'Row D (no TTA) minus Row E'), ('d_tta_vs_e', 'Row D (+TTA) minus Row E')]:
    diff_all = d5_data['paired_differences'][key]['all_images']
    diff_no_outlier = d5_data['paired_differences'][key]['without_065_HC']
    print(f"{label} (All 999 images): {diff_all['diff_point']:+.4f} mm (95% CI: [{diff_all['ci_95'][0]:.4f}, {diff_all['ci_95'][1]:.4f}])")
    print(f"{label} (Excl. 065_HC.png): {diff_no_outlier['diff_point']:+.4f} mm (95% CI: [{diff_no_outlier['ci_95'][0]:.4f}, {diff_no_outlier['ci_95'][1]:.4f}])")

=== Row D vs Row E 5-Fold Cross-Validation Summary ===
Total pooled images: 999
Row D (no TTA) Pooled HC MAE:   2.2688 mm, BPD MAE: 0.7560 mm
Row D (+TTA)    Pooled HC MAE:   2.2323 mm, BPD MAE: 0.7560 mm
Row E           Pooled HC MAE:   2.3019 mm, BPD MAE: 0.7535 mm

--- Per-Fold Comparison ---
 Fold   N  D HC MAE  D+TTA HC MAE  E HC MAE  D BPD MAE  D+TTA BPD MAE  E BPD MAE
    1 206    2.1662        2.1578    2.4817     0.6874         0.6821     0.6982
    2 199    2.2055        2.3016    2.2939     0.7359         0.8030     0.8052
    3 208    2.3151        2.2983    2.1582     0.7246         0.7174     0.6586
    4 194    2.3044        2.3258    2.2446     0.8484         0.8319     0.7726
    5 192    2.3582        2.0746    2.3311     0.7913         0.7517     0.8430

--- Paired Difference (D - E) in HC MAE with Patient-Level Bootstrap 95% CI ---
Row D (no TTA) minus Row E (All 999 images): -0.0332 mm (95% CI: [-0.1922, 0.1272])
Row D (no TTA) minus Row E (Excl. 065_HC.png): -0.03

In [3]:
from pathlib import Path

results_dir = None
for candidate in [
    Path('../../docs/results/hc18_ladder'),
    Path('work/docs/results/hc18_ladder'),
    Path('docs/results/hc18_ladder'),
    Path('/kaggle/working/out'),
]:
    if (candidate / 'd_5fold_table.md').exists():
        results_dir = candidate
        break

if results_dir is not None:
    print((results_dir / 'd_5fold_table.md').read_text(encoding='utf-8'))

# HC18 Row D (ResNet-34 U-Net 224) 5-Fold Benchmark vs Row E

Evaluated on all 999 images across all 5 grouped folds (`make_grouped_folds`, seed 42) with inner 10% patient-isolation validation.

## Per-Fold Performance

| Fold | N Images | D HC MAE (mm) | D+TTA HC MAE (mm) | E HC MAE (mm) | D BPD MAE (mm) | D+TTA BPD MAE (mm) | E BPD MAE (mm) |
|:---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 206 | 2.1662 | 2.1578 | 2.4817 | 0.6874 | 0.6821 | 0.6982 |
| 2 | 199 | 2.2055 | 2.3016 | 2.2939 | 0.7359 | 0.8030 | 0.8052 |
| 3 | 208 | 2.3151 | 2.2983 | 2.1582 | 0.7246 | 0.7174 | 0.6586 |
| 4 | 194 | 2.3044 | 2.3258 | 2.2446 | 0.8484 | 0.8319 | 0.7726 |
| 5 | 192 | 2.3582 | 2.0746 | 2.3311 | 0.7913 | 0.7517 | 0.8430 |
| **Pooled** | **999** | **2.2688** | **2.2323** | **2.3019** | **0.7560** | **0.7560** | **0.7535** |

## Pooled Metrics Summary

| Model / Approach | Input Res | TTA | Pooled HC MAE (mm) | Pooled HC RMSE (mm) | Pooled BPD MAE (mm) | Pooled BPD RMSE (mm) |
|:---|:---:|:---:|---: